# PricePilot AI — Price Prediction Model

## Milestone 2: Model Training, Evaluation, Price Prediction & Groq Integration

### Goal
Train a machine-learning model that predicts product price from historical business features and expose the trained model to the PricePilot AI backend/frontend.

### This notebook covers
1. Problem statement
2. Dataset loading
3. Data cleaning
4. Exploratory analysis
5. Feature engineering
6. Train/test split
7. XGBoost price prediction model
8. Model evaluation — MAE, RMSE and R²
9. Sample price prediction
10. Model saving
11. Price recommendation simulation
12. Groq API connection
13. Using Groq to explain the pricing result

> The notebook is documentation + experimentation. The production API should load the saved model from the backend `models/` folder.


## 1. Problem Statement

PricePilot AI needs a price prediction component that can learn the relationship between:

- historical price
- quantity / demand
- discount
- category
- date/seasonality
- customer/order information where available

The model predicts a numerical **price**. The recommendation engine can then compare candidate prices using expected demand/revenue and select a recommended price.

### Evaluation metrics

- **MAE (Mean Absolute Error):** average absolute prediction error.
- **RMSE (Root Mean Squared Error):** penalizes larger errors more strongly.
- **R²:** how much variance in the target is explained by the model.


## 2. Imports

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

print("Libraries loaded successfully.")


ModuleNotFoundError: No module named 'xgboost'

## 3. Locate the PricePilot AI dataset

This notebook uses the Amazon dataset because it contains a direct price field (`UnitPrice`) together with demand and discount variables.

If your project folder is opened from another working directory, the code searches upward for the `data` folder.


In [ ]:
ROOT = Path.cwd()

for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "data" / "raw" / "Amazon.csv").exists():
        ROOT = candidate
        break

DATA_FILE = ROOT / "data" / "raw" / "Amazon.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Dataset:", DATA_FILE)
print("Model directory:", MODEL_DIR)


## 4. Load dataset

In [ ]:
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


## 5. Basic data inspection

In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing"))

print("Duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))


In [ ]:
numeric_candidates = [
    "Quantity", "UnitPrice", "Discount", "Tax",
    "ShippingCost", "TotalAmount"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "OrderDate" in df.columns:
    df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")

display(df[numeric_candidates].describe(include="all"))


## 6. Data cleaning

For price prediction we remove records where the target price is missing or non-positive.

The model is trained only on usable observations.


In [ ]:
required = ["UnitPrice", "Quantity", "Discount", "Category", "OrderDate"]

missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

clean = df.copy()

clean = clean[
    clean["UnitPrice"].notna() &
    (clean["UnitPrice"] > 0) &
    clean["Quantity"].notna() &
    (clean["Quantity"] > 0) &
    clean["OrderDate"].notna()
].copy()

clean["Discount"] = clean["Discount"].fillna(0)
clean["Category"] = clean["Category"].fillna("Unknown").astype(str)

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(clean))


## 7. Feature engineering

We create time features because pricing can change with seasonality.

### Features
- Quantity
- Discount
- Tax
- ShippingCost
- TotalAmount
- Category
- Month
- Day of week
- Day of month
- Year

### Target
`UnitPrice`


In [ ]:
clean["month"] = clean["OrderDate"].dt.month
clean["day_of_week"] = clean["OrderDate"].dt.dayofweek
clean["day_of_month"] = clean["OrderDate"].dt.day
clean["year"] = clean["OrderDate"].dt.year
clean["quarter"] = clean["OrderDate"].dt.quarter

feature_cols = [
    "Quantity",
    "Discount",
    "Tax",
    "ShippingCost",
    "TotalAmount",
    "Category",
    "month",
    "day_of_week",
    "day_of_month",
    "year",
    "quarter",
]

target_col = "UnitPrice"

X = clean[feature_cols].copy()
y = clean[target_col].copy()

print("Features:", feature_cols)
print("Target:", target_col)
print("X shape:", X.shape)
print("y shape:", y.shape)


## 8. Train / test split

80% of the data is used for training and 20% for final evaluation.

`random_state=42` makes the experiment reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))


## 9. Preprocessing pipeline

Numerical missing values are filled using the median.

`Category` is converted to machine-readable one-hot encoded features.


In [ ]:
numeric_features = [
    "Quantity",
    "Discount",
    "Tax",
    "ShippingCost",
    "TotalAmount",
    "month",
    "day_of_week",
    "day_of_month",
    "year",
    "quarter",
]

categorical_features = ["Category"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## 10. Train XGBoost price prediction model

XGBoost is a gradient-boosted tree model and is suitable for tabular business data.

The model learns:

`business features → historical price`


In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=2
)

price_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

price_pipeline.fit(X_train, y_train)

print("XGBoost price prediction model trained successfully.")


## 11. Model evaluation

We calculate:

### MAE
Lower is better.

### RMSE
Lower is better.

### R²
Closer to 1 generally means better explanatory performance.


In [ ]:
y_pred = price_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = {
    "MAE": float(mae),
    "RMSE": float(rmse),
    "R2": float(r2),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test))
}

print(json.dumps(metrics, indent=2))


## 12. Actual vs predicted price

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
}).reset_index(drop=True)

display(comparison.head(20))


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Price Prediction — Actual vs Predicted")

line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
plt.plot([line_min, line_max], [line_min, line_max])
plt.tight_layout()
plt.show()


## 13. Residual analysis

In [ ]:
residuals = y_test.values - y_pred

plt.figure(figsize=(10,5))
plt.hist(residuals, bins=40)
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")
plt.title("Price Model Residual Distribution")
plt.tight_layout()
plt.show()

print("Mean residual:", float(np.mean(residuals)))


## 14. Example: predict a product price

The following function prepares one product/business observation and returns the predicted price.

In the production backend, the same saved pipeline is loaded and this logic is called by an API endpoint.


In [ ]:
def predict_price(
    quantity,
    discount,
    tax,
    shipping_cost,
    total_amount,
    category,
    order_date
):
    date = pd.to_datetime(order_date)

    sample = pd.DataFrame([{
        "Quantity": quantity,
        "Discount": discount,
        "Tax": tax,
        "ShippingCost": shipping_cost,
        "TotalAmount": total_amount,
        "Category": str(category),
        "month": date.month,
        "day_of_week": date.dayofweek,
        "day_of_month": date.day,
        "year": date.year,
        "quarter": date.quarter,
    }])

    prediction = float(price_pipeline.predict(sample)[0])
    return max(0.0, prediction)


example_price = predict_price(
    quantity=5,
    discount=0.10,
    tax=20,
    shipping_cost=10,
    total_amount=500,
    category=str(clean["Category"].iloc[0]),
    order_date=clean["OrderDate"].iloc[0]
)

print("Predicted price:", round(example_price, 2))


## 15. Save the trained model

The complete preprocessing + XGBoost pipeline is saved as one file.

This is important because the backend must perform exactly the same preprocessing during inference.


In [ ]:
MODEL_FILE = MODEL_DIR / "price_prediction_model.pkl"

joblib.dump(price_pipeline, MODEL_FILE)

metrics_file = MODEL_DIR / "price_model_metrics.json"
metrics_file.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Saved model:", MODEL_FILE)
print("Saved metrics:", metrics_file)


## 16. Verify saved model can be loaded

In [ ]:
loaded_model = joblib.load(MODEL_FILE)

test_prediction = loaded_model.predict(X_test.head(1))[0]

print("Loaded model successfully.")
print("Verification prediction:", float(test_prediction))


# 17. Optimal Price Recommendation — Revenue Simulation

A **price prediction model** and an **optimal price recommendation engine** are different:

- Price prediction predicts a price from business features.
- Recommendation evaluates candidate prices and selects the price with the best estimated revenue.

For Milestone 2, candidate prices can be simulated around the current price.

The simple local elasticity simulation below is designed as a transparent baseline. It should be presented as a recommendation heuristic, not as a causal proof.


In [ ]:
def recommend_price(current_price, base_units, elasticity=-1.2, steps=21, span=0.20):
    current_price = float(current_price)
    base_units = max(float(base_units), 0.0)

    candidate_prices = np.linspace(
        current_price * (1 - span),
        current_price * (1 + span),
        steps
    )

    rows = []

    for price in candidate_prices:
        ratio = price / current_price
        expected_units = base_units * (ratio ** elasticity)
        expected_revenue = price * expected_units

        rows.append({
            "candidate_price": price,
            "expected_units": expected_units,
            "expected_revenue": expected_revenue
        })

    result = pd.DataFrame(rows)
    best = result.loc[result["expected_revenue"].idxmax()]

    return result, best

current_price = float(clean["UnitPrice"].median())
base_units = float(clean["Quantity"].median())

simulation, best = recommend_price(
    current_price=current_price,
    base_units=base_units
)

print("Current price:", round(current_price, 2))
print("Recommended price:", round(float(best["candidate_price"]), 2))
print("Expected units:", round(float(best["expected_units"]), 2))
print("Expected revenue:", round(float(best["expected_revenue"]), 2))

display(simulation)


## 18. Groq API Connection

Groq is used as an **AI explanation layer**, not as the numerical price prediction model.

The ML model produces the numeric prediction/recommendation. Groq can convert those results into a human-readable business explanation for the dashboard.

### Keep the API key in `.env`
Never hard-code the API key in this notebook or commit it to GitHub.


In [ ]:
# If needed, install once from the project terminal:
# python -m pip install groq python-dotenv

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("GROQ_API_KEY not found.")
    print("Add it to backend/.env before running the Groq cells.")
else:
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq client connected successfully.")


## 19. Groq pricing explanation

This example sends only the numerical/business result to Groq and asks for a short explanation.

Do **not** send secrets or unnecessary customer PII.


In [ ]:
def explain_pricing_with_groq(
    current_price,
    recommended_price,
    expected_units,
    expected_revenue,
    mae,
    rmse
):
    if not GROQ_API_KEY:
        return "Groq API key is not configured."

    prompt = f'''
You are a pricing analyst for PricePilot AI.

Explain this pricing result in simple business language.

Current price: {current_price:.2f}
Recommended price: {recommended_price:.2f}
Expected units: {expected_units:.2f}
Expected revenue: {expected_revenue:.2f}
Price model MAE: {mae:.4f}
Price model RMSE: {rmse:.4f}

Give:
1. recommended action
2. expected business impact
3. one important caution about using an ML recommendation
Keep it concise.
'''

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a business pricing analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content

if GROQ_API_KEY:
    explanation = explain_pricing_with_groq(
        current_price=current_price,
        recommended_price=float(best["candidate_price"]),
        expected_units=float(best["expected_units"]),
        expected_revenue=float(best["expected_revenue"]),
        mae=mae,
        rmse=rmse
    )
    print(explanation)


# 20. Backend Integration

The production backend should load:

`models/price_prediction_model.pkl`

Then expose an endpoint such as:

`POST /api/price/predict`

Example request:

```json
{
  "quantity": 5,
  "discount": 0.10,
  "tax": 20,
  "shipping_cost": 10,
  "total_amount": 500,
  "category": "Electronics",
  "order_date": "2024-12-01"
}
```

Example response:

```json
{
  "predicted_price": 98.42,
  "model": "XGBoost",
  "mae": 0.XX,
  "rmse": 0.XX
}
```

The frontend calls this endpoint and displays the predicted/recommended price.

### Important architecture

```text
Dataset
   ↓
Preprocessing
   ↓
XGBoost Price Model
   ↓
MAE / RMSE / R²
   ↓
Saved .pkl model
   ↓
FastAPI endpoint
   ↓
Frontend
   ↓
Price + Revenue + Explanation

Groq API
   ↓
Human-readable explanation layer
```


# 21. Milestone 2 Completion Checklist

- [x] Price prediction model
- [x] XGBoost training
- [x] Train/test split
- [x] MAE evaluation
- [x] RMSE evaluation
- [x] R² evaluation
- [x] Actual vs predicted visualization
- [x] Residual analysis
- [x] Model saved as `.pkl`
- [x] Price recommendation simulation
- [x] Groq API connection
- [x] Groq explanation layer
- [ ] Connect the saved model to the FastAPI production endpoint
- [ ] Connect the endpoint to the frontend

The final two items belong to the project backend/frontend code rather than notebook experimentation.
